In [1]:
import pandas as pd
import numpy as np
import torch
import faiss
from datasets import load_dataset
from sentence_transformers import SentenceTransformer

from utils import cosine_search

/Users/bahloulia/Downloads/agentic_software/RAG-Applications/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Hotel Finder — Semantic Search

## What we're going to do

![What we're going to do](../images/ch04__image003.png)

In [2]:
dataset = load_dataset("traversaal-ai-hackathon/hotel_datasets")
df = pd.DataFrame(dataset['train'])
df.head()

,hotel_name,hotel_description,review_title,review_text,rate,tripdate,hotel_url,hotel_image,price_range,rating_value,review_count,street_address,locality,country
0,Romance Istanbul Hotel,Romance Istanbul Hotel has 39 rooms.Every room...,"An exceptional boutique hotel, great value for...",NaN,NaN,February 2020,https://www.tripadvisor.com/Hotel_Review-g2939...,https://media-cdn.tripadvisor.com/media/photo-...,$ (Based on Average Nightly Rates for a Standa...,5.0,4023,Hudavendigar Cd. No:5 Sirkeci,Istanbul,Turkiye
1,Romance Istanbul Hotel,Romance Istanbul Hotel has 39 rooms.Every room...,You can’t get better than this.,NaN,NaN,March 2021,https://www.tripadvisor.com/Hotel_Review-g2939...,https://media-cdn.tripadvisor.com/media/photo-...,$ (Based on Average Nightly Rates for a Standa...,5.0,4023,Hudavendigar Cd. No:5 Sirkeci,Istanbul,Turkiye
2,Romance Istanbul Hotel,Romance Istanbul Hotel has 39 rooms.Every room...,Exceeds all expectations,NaN,NaN,March 2021,https://www.tripadvisor.com/Hotel_Review-g2939...,https://media-cdn.tripadvisor.com/media/photo-...,$ (Based on Average Nightly Rates for a Standa...,5.0,4023,Hudavendigar Cd. No:5 Sirkeci,Istanbul,Turkiye
3,Romance Istanbul Hotel,Romance Istanbul Hotel has 39 rooms.Every room...,"Great Location, Fantastic Accommodations",NaN,NaN,August 2021,https://www.tripadvisor.com/Hotel_Review-g2939...,https://media-cdn.tripadvisor.com/media/photo-...,$ (Based on Average Nightly Rates for a Standa...,5.0,4023,Hudavendigar Cd. No:5 Sirkeci,Istanbul,Turkiye
4,Romance Istanbul Hotel,Romance Istanbul Hotel has 39 rooms.Every room...,Perfection. It is all in the details.,NaN,NaN,June 2021,https://www.tripadvisor.com/Hotel_Review-g2939...,https://media-cdn.tripadvisor.com/media/photo-...,$ (Based on Average Nightly Rates for a Standa...,5.0,4023,Hudavendigar Cd. No:5 Sirkeci,Istanbul,Turkiye


In [3]:
df_paris = df.loc[df.locality == 'Paris']
df_paris.head()
df_paris.hotel_name.value_counts()

hotel_name
Hotel Malte - Astotel                  40
Hotel Astoria - Astotel                40
Novotel Paris Les Halles               40
La Maison Favart                       40
Grand Hotel du Palais Royal            40
Hotel Maison Mere                      40
Hotel des Arts - Montmartre            40
Hotel Joke - Astotel                   40
Passy Eiffel Hotel                     40
Best Western Plus La Demeure           40
Hotel 34B - Astotel                    40
Hotel La Comtesse                      40
Cler Hotel                             40
Hotel Marignan Champs-Elysees          40
Hotel Piapia                           40
Hotel Moliere                          40
citizenM Paris Champs-Elysees          40
Hotel du Danube Saint Germain          40
Hotel B55                              40
Hotel Europe Saint Severin             40
Citadines Tour Eiffel Paris            40
Pullman Paris Eiffel Tower Hotel       40
Hotel Tourisme Avenue                  40
Grand Hotel Malher     

In [4]:
model = SentenceTransformer("all-MiniLM-L6-v2")

if torch.cuda.is_available():
    model = model.to('cuda')
    print("CUDA is available. The model has been moved to GPU.")
else:
    print("CUDA is not available. The model will run on CPU.")

CUDA is not available. The model will run on CPU.


In [5]:
reviews = df_paris['review_text'].tolist()
review_embeddings = model.encode(reviews, show_progress_bar=True)
print(f"Embeddings shape: {review_embeddings.shape}") 

Batches:   0%|          | 0/38 [00:00<?, ?it/s]

Batches:   3%|▎         | 1/38 [00:01<00:37,  1.00s/it]

Batches:   5%|▌         | 2/38 [00:01<00:32,  1.12it/s]

Batches:   8%|▊         | 3/38 [00:02<00:33,  1.04it/s]

Batches:  11%|█         | 4/38 [00:03<00:33,  1.03it/s]

...

Batches:  95%|█████████▍| 36/38 [00:19<00:00,  3.42it/s]

Batches:  97%|█████████▋| 37/38 [00:19<00:00,  3.70it/s]

Batches: 100%|██████████| 38/38 [00:19<00:00,  4.35it/s]

Batches: 100%|██████████| 38/38 [00:19<00:00,  1.95it/s]

Embeddings shape: (1200, 384)


## BI-Encoder

![BI-Encoder](../images/ch04__image009.png)

In [6]:
query = " Hotel near the Louvre with great food nearby."
query_embedding = model.encode([query])

k = 5  # Number of similar reviews to retrieve
indices, distances = cosine_search(query_embedding, review_embeddings, k)
print(f"Query: {query}")
print("Top hotel with similar reviews matching the query:")
for i, (idx, distance) in enumerate(zip(indices, distances), 1):
    print(f"{i}. {df_paris.iloc[idx]['hotel_name']}")
    print(f"Review: {df_paris.iloc[idx]['review_text']}")
    print(f"Distance: {distance:.4f}")
    print()

Query:  Hotel near the Louvre with great food nearby.
Top hotel with similar reviews matching the query:
1. Hotel Malte - Astotel
Review: Brilliant hotel, clean, comfortable, great location just a short walk to the Louvre. The open bar in the afternoon is fantastic, we could refresh with coffee and pastries throughout the day at any of their locations, which is such a brilliant idea. Staff were friendly and welcoming. Cannot fault the hotel in any way. 
Distance: 0.7819

2. Grand Hotel du Palais Royal
Review: The hotel is in a great location, near the Louvre and walking distance to restaurants.  The staff at this hotel go above and beyond, very professional and with great suggestions/recommendations. Rooms are very clean and the pillows/bedding super comfortable. Will definitely return!
Distance: 0.7645

3. Hotel Square Louvois
Review: Outstanding boutique hotel, friendly and welcoming staff, beautifully decorated rooms and public spaces, located on a fairly quiet street but with plent

## Introduction to FAISS and Vector Databases

The `cosine_search` function above works, but it's a brute-force NumPy loop over every review embedding on every query — fine for 1,200 reviews, but it scales linearly, so it gets slow once the corpus grows to tens of thousands of hotels across many cities.

**FAISS** (Facebook AI Similarity Search) is a library purpose-built for this: fast similarity search over large collections of vectors, implemented in C++ and vectorized/multithreaded under the hood, so the same nearest-neighbor search runs far faster than the equivalent hand-written NumPy. It supports both exact search (`IndexFlatIP`, used below — same math as `cosine_search`, just a faster implementation) and *approximate* indexes (IVF, HNSW) that trade a small amount of accuracy for sub-linear search time on millions of vectors, which is what you'd reach for once brute-force stops being fast enough.

FAISS itself is just an in-memory index, not a full **vector database** — it doesn't handle persistence, metadata filtering, or a query API out of the box the way Chroma (used in the other notebooks in this repo) or Pinecone/Qdrant do. It's the retrieval *algorithm* underneath those systems, useful here as a lighter-weight way to see what a vector store is actually doing before reaching for a full database.

![Hotel Finder pipeline with FAISS](../images/ch04__image013.png)

In [7]:
review_embeddings = review_embeddings.astype('float32')

review_embeddings_normalized = review_embeddings / np.linalg.norm(review_embeddings, axis=1, keepdims=True)

index = faiss.IndexFlatIP(review_embeddings_normalized.shape[1])
index.add(review_embeddings_normalized)
query = "Hotel near the Louvre with great food nearby."
query_embedding = model.encode([query]).astype('float32')
query_embedding_normalized = query_embedding / np.linalg.norm(query_embedding)
k = 5
distances, indices = index.search(query_embedding, k)
print(f"Query: {query}")
print("Top hotel with similar reviews using FAISS:")
for i, (idx, distance) in enumerate(zip(indices[0], distances[0]), 1):
    print(f"{i}. {df_paris.iloc[idx]['hotel_name']}")
    print(f"Review: {df_paris.iloc[idx]['review_text']}")
    print(f"Distance: {distance:.4f}")
    print()

Query: Hotel near the Louvre with great food nearby.
Top hotel with similar reviews using FAISS:
1. Hotel Malte - Astotel
Review: Brilliant hotel, clean, comfortable, great location just a short walk to the Louvre. The open bar in the afternoon is fantastic, we could refresh with coffee and pastries throughout the day at any of their locations, which is such a brilliant idea. Staff were friendly and welcoming. Cannot fault the hotel in any way. 
Distance: 0.7819

2. Grand Hotel du Palais Royal
Review: The hotel is in a great location, near the Louvre and walking distance to restaurants.  The staff at this hotel go above and beyond, very professional and with great suggestions/recommendations. Rooms are very clean and the pillows/bedding super comfortable. Will definitely return!
Distance: 0.7645

3. Hotel Square Louvois
Review: Outstanding boutique hotel, friendly and welcoming staff, beautifully decorated rooms and public spaces, located on a fairly quiet street but with plenty of bis